In [0]:
%pip install torch

In [0]:
from copy import deepcopy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

transactions = spark.table('fraud_transactions_for_detection').toPandas()
print('Rows and columns:', transactions.shape)
transactions.head()


In [0]:
transactions = transactions.reset_index(drop=True)
assert transactions["Transaction_ID"].is_unique
assert transactions["User_ID"].notna().all()
assert transactions["Fraudulent"].notna().all()
assert transactions["Fraudulent"].isin([0, 1]).all()

# First split 20% test
first_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

development_idx, test_idx = next(
    first_split.split(
        transactions,
        groups=transactions["User_ID"]
    )
)

development = transactions.iloc[development_idx].copy()
test = transactions.iloc[test_idx].copy()

# Second split: 20% validation
second_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, validation_idx = next(
    second_split.split(
        development,
        groups=development["User_ID"]
    )
)

train = development.iloc[train_idx].copy()
validation = development.iloc[validation_idx].copy()

In [0]:
split_summary = pd.DataFrame([
    {
        "set": name,
        "transactions": len(frame),
        "users": frame["User_ID"].nunique(),
        "fraud_count": frame["Fraudulent"].sum(),
        "fraud_rate_pct": 100 * frame["Fraudulent"].mean()
    }
    for name, frame in [
        ("Training", train),
        ("Validation", validation),
        ("Test", test)
    ]
])

display(split_summary)

# No Duplicate Users.
assert set(train["User_ID"]).isdisjoint(validation["User_ID"])
assert set(train["User_ID"]).isdisjoint(test["User_ID"])
assert set(validation["User_ID"]).isdisjoint(test["User_ID"])

# Every transaction should belong to exactly one set.
assert len(train) + len(validation) + len(test) == len(transactions)
assert transactions["Transaction_ID"].notna().all(), \
    "Missing transaction IDs found."

In [0]:
# Legitimate only training set
train_legitimate = train.loc[
    train["Fraudulent"] == 0
].copy()
print("Legitimate training transactions:", len(train_legitimate))

In [0]:
# Determine which features have missing values
missing_summary = (
    train_legitimate.isna()
    .sum()
    .rename("missing_count")
    .rename_axis("feature")
    .reset_index()
)

missing_summary["missing_pct"] = (
    100 * missing_summary["missing_count"] / len(train_legitimate)
)

display(
    missing_summary.sort_values(
        "missing_count",
        ascending=False
    )
)

In [0]:
numeric_features = [
    "Transaction_Amount",
    "Time_of_Transaction",
    "Account_Age",
    "Number_of_Transactions_Last_24H"
]

# Select numerical features and ensure they are numeric.
train_numeric = train_legitimate[numeric_features].astype(float)

# Learn one median per feature, using legitimate training data only.
numeric_medians = train_numeric.median()

# Stop if a feature has no observed training values.
assert numeric_medians.notna().all()

# Fill missing values before fitting the scaler.
train_numeric_filled = train_numeric.fillna(numeric_medians)

# Learn the mean and standard deviation of each feature.
numeric_scaler = StandardScaler()
numeric_scaler.fit(train_numeric_filled)

In [0]:
def prepare_numeric(frame):
    values = frame[numeric_features].astype(float)

    # 1 = originally observed; 0 = originally missing.
    observed_mask = values.notna().to_numpy(dtype=np.float32)

    filled = values.fillna(numeric_medians)

    # Apply the existing scaler
    scaled = numeric_scaler.transform(filled).astype(np.float32)

    # Separate indicators for the encoder: 1 means missing.
    missing_flags = 1.0 - observed_mask

    return scaled, observed_mask, missing_flags

In [0]:
#Apply the settings to the training/test/validation sets
train_num, train_num_mask, train_num_missing = prepare_numeric(
    train_legitimate
)

val_num, val_num_mask, val_num_missing = prepare_numeric(
    validation
)

test_num, test_num_mask, test_num_missing = prepare_numeric(
    test
)

# Check that it doesnt become infinite.
for values in [train_num, val_num, test_num]:
    assert np.isfinite(values).all()

print("Training:", train_num.shape)
print("Validation:", val_num.shape)
print("Test:", test_num.shape)

In [0]:
categorical_features = [
    "Transaction_Type",
    "Device_Used",
    "Location",
    "Payment_Method"
]

MISSING_CATEGORY = "__MISSING__"
UNSEEN_CATEGORY = "__UNSEEN__"

category_maps = {}

for feature in categorical_features:
    observed_categories = sorted(
        train_legitimate[feature]
        .dropna()
        .astype(str)
        .unique()
        .tolist()
    )

    assert MISSING_CATEGORY not in observed_categories
    assert UNSEEN_CATEGORY not in observed_categories

    categories = [
        MISSING_CATEGORY,
        UNSEEN_CATEGORY,
        *observed_categories
    ]

    category_maps[feature] = {
        category: index
        for index, category in enumerate(categories)
    }

category_sizes = [
    len(category_maps[feature])
    for feature in categorical_features
]

display(pd.DataFrame({
    "feature": categorical_features,
    "number_of_categories": category_sizes
}))

In [0]:
def prepare_categorical(frame):
    encoded_columns = []
    observed_columns = []

    for feature in categorical_features:
        values = frame[feature].astype("string")
        mapping = category_maps[feature]

        codes = (
            values.fillna(MISSING_CATEGORY)
            .map(mapping)
            .fillna(mapping[UNSEEN_CATEGORY])
            .to_numpy(dtype=np.int64)
        )

        # Reconstruction targets must be observed.
        observed = (
            values.notna().to_numpy()
            & (codes != mapping[UNSEEN_CATEGORY])
        )

        encoded_columns.append(codes)
        observed_columns.append(observed)

    return (
        np.column_stack(encoded_columns),
        np.column_stack(observed_columns).astype(np.float32)
    )


train_cat, train_cat_mask = prepare_categorical(train_legitimate)
val_cat, val_cat_mask = prepare_categorical(validation)
test_cat, test_cat_mask = prepare_categorical(test)

In [0]:
def assemble_encoder_inputs(numeric_values, missing_flags, categories):
    one_hot_parts = []

    for column, size in enumerate(category_sizes):
        identity_matrix = np.eye(size, dtype=np.float32)
        category_codes = categories[:, column]
        one_hot_values = identity_matrix[category_codes]

        one_hot_parts.append(one_hot_values)

    return np.concatenate(
        [numeric_values, missing_flags, *one_hot_parts],
        axis=1
    ).astype(np.float32)

X_train = assemble_encoder_inputs(
    train_num, train_num_missing, train_cat
)

X_val = assemble_encoder_inputs(
    val_num, val_num_missing, val_cat
)

X_test = assemble_encoder_inputs(
    test_num, test_num_missing, test_cat
)
expected_columns = (
    len(numeric_features) * 2 + sum(category_sizes)
)

for inputs in [X_train, X_val, X_test]:
    assert inputs.shape[1] == expected_columns

print("Encoder training inputs:", X_train.shape)


In [0]:
class TransactionVAE(nn.Module):
    def __init__(
        self,
        input_dim,
        numeric_dim,
        category_sizes,
        latent_dim=4
    ):
        super().__init__()

        # h = Wx + b, #64, 32 are hyperparameters
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU()
        )

        # Parameters of q(z | x).
        self.latent_mean = nn.Linear(32, latent_dim)
        self.latent_log_variance = nn.Linear(32, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU()
        )

        # Separate outputs for numerical and categorical features.
        self.numeric_output = nn.Linear(64, numeric_dim)

        self.categorical_outputs = nn.ModuleList([
            nn.Linear(64, size)
            for size in category_sizes
        ])

    # Encoding function
    def encode(self, x):
        hidden = self.encoder(x)
        mean = self.latent_mean(hidden)
        log_variance = self.latent_log_variance(hidden)
        return mean, log_variance

    # Sampling using reparamaterization trick
    def reparameterize(self, mean, log_variance):
        standard_deviation = torch.exp(0.5 * log_variance)
        noise = torch.randn_like(standard_deviation) 
        return mean + standard_deviation * noise # z = mu + sigma * epsilon, epsilon ~ N(0, 1)

    # Decoding function
    def decode(self, z):
        hidden = self.decoder(z)

        numeric_reconstruction = self.numeric_output(hidden)

        categorical_logits = [
            output_layer(hidden)
            for output_layer in self.categorical_outputs
        ]

        return numeric_reconstruction, categorical_logits

    def forward(self, x, sample=True): #Combines everything into the full forward pass.
        mean, log_variance = self.encode(x)

        if sample:
            z = self.reparameterize(mean, log_variance)
        else:
            z = mean

        numeric_reconstruction, categorical_logits = self.decode(z)

        return (
            numeric_reconstruction,
            categorical_logits,
            mean,
            log_variance
        )

In [0]:
torch.manual_seed(42)

model = TransactionVAE(
    input_dim=X_train.shape[1],
    numeric_dim=len(numeric_features),
    category_sizes=category_sizes,
    latent_dim=4
)

example_batch = torch.from_numpy(X_train[:8])

with torch.no_grad():
    numeric_reconstruction, categorical_logits, mean, log_variance = (
        model(example_batch)
    )

print("Numerical reconstruction:", numeric_reconstruction.shape)
print("Latent mean:", mean.shape)
print("Latent log variance:", log_variance.shape)

for feature, logits in zip(categorical_features, categorical_logits):
    print(f"{feature} output:", logits.shape)

In [0]:


def vae_loss(
    outputs,
    numeric_targets,
    categorical_targets,
    numeric_mask,
    categorical_mask,
    beta=1.0
):
    (
        numeric_reconstruction,
        categorical_logits,
        mean,
        log_variance
    ) = outputs

    # Numerical reconstruction, ignores originally missing values.
    numeric_errors = 0.5 * (
        numeric_reconstruction - numeric_targets
    ).pow(2)

    numeric_loss = (
        numeric_errors * numeric_mask
    ).sum(dim=1)

    # Categorical reconstruction, ignores missing or unseen categories.
    categorical_loss = torch.zeros_like(numeric_loss)

    for column, logits in enumerate(categorical_logits):
        feature_loss = F.cross_entropy(
            logits,
            categorical_targets[:, column],
            reduction="none"
        )

        categorical_loss = (
            categorical_loss
            + feature_loss * categorical_mask[:, column]
        )

    # KL divergence from the encoder distribution to N(0, I).
    kl_loss = -0.5 * (
        1
        + log_variance
        - mean.pow(2)
        - log_variance.exp()
    ).sum(dim=1)

    reconstruction_loss = numeric_loss + categorical_loss

    total_per_transaction = reconstruction_loss + beta * kl_loss

    # One scalar for optimization
    return {
        "loss": total_per_transaction.mean(),
        "numeric": numeric_loss.mean(),
        "categorical": categorical_loss.mean(),
        "kl": kl_loss.mean(),
        "reconstruction_per_transaction": reconstruction_loss
    }

In [0]:
def make_dataset(inputs, numeric, categories, numeric_mask, category_mask):
    return TensorDataset(
        torch.from_numpy(inputs).float(),
        torch.from_numpy(numeric).float(),
        torch.from_numpy(categories).long(),
        torch.from_numpy(numeric_mask).float(),
        torch.from_numpy(category_mask).float()
    )


training_dataset = make_dataset(
    X_train,
    train_num,
    train_cat,
    train_num_mask,
    train_cat_mask
)

# Use legitimate validation transactions for monitoring VAE training.
validation_legitimate = (
    validation["Fraudulent"].to_numpy() == 0
)

validation_dataset = make_dataset(
    X_val[validation_legitimate],
    val_num[validation_legitimate],
    val_cat[validation_legitimate],
    val_num_mask[validation_legitimate],
    val_cat_mask[validation_legitimate]
)

assert len(training_dataset) > 0
assert len(validation_dataset) > 0

train_loader = DataLoader(
    training_dataset,
    batch_size=256,
    shuffle=True,
    generator=torch.Generator().manual_seed(42)
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=256,
    shuffle=False
)

In [0]:
torch.manual_seed(42)

# This small model can run on CPU.
model = TransactionVAE(
    input_dim=X_train.shape[1],
    numeric_dim=len(numeric_features),
    category_sizes=category_sizes,
    latent_dim=4
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

BETA = 1.0
MAX_EPOCHS = 50
PATIENCE = 7

In [0]:
def run_epoch(loader, training):
    model.train(training)

    totals = {
        "loss": 0.0,
        "numeric": 0.0,
        "categorical": 0.0,
        "kl": 0.0
    }

    transaction_count = 0

    with torch.set_grad_enabled(training):
        for inputs, numeric, categories, numeric_mask, category_mask in loader:

            if training:
                optimizer.zero_grad(set_to_none=True)

            outputs = model(inputs, sample=True)

            losses = vae_loss(
                outputs=outputs,
                numeric_targets=numeric,
                categorical_targets=categories,
                numeric_mask=numeric_mask,
                categorical_mask=category_mask,
                beta=BETA
            )

            if not torch.isfinite(losses["loss"]).item():
                raise RuntimeError(
                    "Non-finite loss encountered. Stop and inspect the model."
                )

            if training:
                losses["loss"].backward()

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=5.0,
                    error_if_nonfinite=True
                )

                optimizer.step()

            batch_size = inputs.shape[0]
            transaction_count += batch_size

            for name in totals:
                totals[name] += losses[name].detach().item() * batch_size

    return {
        name: total / transaction_count
        for name, total in totals.items()
    }

In [0]:
best_validation_loss = float("inf")
best_weights = None
best_epoch = None
epochs_without_improvement = 0
history = []

for epoch in range(1, MAX_EPOCHS + 1):
    train_metrics = run_epoch(train_loader, training=True)

    # Reuse the same validation random draws each epoch.
    # fork_rng restores the training random state afterward.
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(123)
        validation_metrics = run_epoch(
            validation_loader,
            training=False
        )

    history.append({
        "epoch": epoch,
        **{
            f"train_{name}": value
            for name, value in train_metrics.items()
        },
        **{
            f"val_{name}": value
            for name, value in validation_metrics.items()
        }
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train: {train_metrics['loss']:.4f} | "
        f"Validation: {validation_metrics['loss']:.4f} | "
        f"Validation KL: {validation_metrics['kl']:.4f}"
    )

    if validation_metrics["loss"] < best_validation_loss - 0.0001:
        best_validation_loss = validation_metrics["loss"]
        best_weights = deepcopy(model.state_dict())
        best_epoch = epoch
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping: validation loss stopped improving.")
        break

# Restore the best checkpoint, rather than using the final epoch.
model.load_state_dict(best_weights)
model.eval()

history_df = pd.DataFrame(history)

print(f"\nBest epoch: {best_epoch}")
print(f"Best validation loss: {best_validation_loss:.4f}")

In [0]:
plt.figure(figsize=(8, 4))

plt.plot(
    history_df["epoch"],
    history_df["train_loss"],
    label="Training"
)

plt.plot(
    history_df["epoch"],
    history_df["val_loss"],
    label="Legitimate validation"
)

plt.xlabel("Epoch")
plt.ylabel("VAE loss")
plt.title("VAE training progress")
plt.legend()
plt.tight_layout()
plt.show()

In [0]:
def calculate_anomaly_scores(
    inputs,
    numeric,
    categories,
    numeric_mask,
    category_mask,
    batch_size=256,
    n_samples=10
):
    dataset = make_dataset(
        inputs,
        numeric,
        categories,
        numeric_mask,
        category_mask
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False
    )

    model.eval()
    scores = []

    # Reproducible sampling without changing the outer random state.
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(1234)

        with torch.no_grad():
            for x, num, cat, num_mask, cat_mask in loader:

                observed_count = (
                    num_mask.sum(dim=1)
                    + cat_mask.sum(dim=1)
                )

                if (observed_count == 0).any().item():
                    raise ValueError(
                        "Cannot score a transaction with no observed features."
                    )

                reconstruction_sum = torch.zeros(
                    x.shape[0],
                    dtype=x.dtype,
                    device=x.device
                )

                for _ in range(n_samples):
                    outputs = model(x, sample=True)

                    losses = vae_loss(
                        outputs,
                        num,
                        cat,
                        num_mask,
                        cat_mask
                    )

                    reconstruction_sum += (
                        losses["reconstruction_per_transaction"]
                    )

                batch_scores = (
                    reconstruction_sum
                    / n_samples
                    / observed_count
                )

                scores.append(batch_scores.cpu().numpy())

    return np.concatenate(scores)

In [0]:
validation_scores = calculate_anomaly_scores(
    X_val,
    val_num,
    val_cat,
    val_num_mask,
    val_cat_mask
)

assert np.isfinite(validation_scores).all()
assert len(validation_scores) == len(validation)

validation_results = validation[
    ["Transaction_ID", "User_ID", "Fraudulent"]
].copy()

validation_results["anomaly_score"] = validation_scores

In [0]:
display(
    validation_results.groupby("Fraudulent")["anomaly_score"]
    .agg(["count", "mean", "median", "std"])
    .reset_index()
)

plt.figure(figsize=(8, 4))

for label, name in [(0, "Legitimate"), (1, "Fraudulent")]:
    scores = validation_results.loc[
        validation_results["Fraudulent"] == label,
        "anomaly_score"
    ]

    plt.hist(
        scores,
        bins=40,
        density=True,
        alpha=0.5,
        label=name
    )

plt.xlabel("Anomaly score")
plt.ylabel("Density")
plt.title("Validation anomaly scores by actual label")
plt.legend()
plt.tight_layout()
plt.show()

In [0]:
ALERT_FRACTION = 0.05

threshold = np.quantile(
    validation_scores,
    1 - ALERT_FRACTION
)

validation_results["predicted_fraud"] = (
    validation_results["anomaly_score"] >= threshold
).astype(int)

actual = validation_results["Fraudulent"].to_numpy()
predicted = validation_results["predicted_fraud"].to_numpy()

tp = int(((predicted == 1) & (actual == 1)).sum())
fp = int(((predicted == 1) & (actual == 0)).sum())
fn = int(((predicted == 0) & (actual == 1)).sum())
tn = int(((predicted == 0) & (actual == 0)).sum())

precision = tp / (tp + fp) if tp + fp else np.nan
recall = tp / (tp + fn) if tp + fn else np.nan
fraud_prevalence = actual.mean()

validation_metrics = pd.DataFrame([{
    "threshold": threshold,
    "alerts": tp + fp,
    "alert_rate_pct": 100 * predicted.mean(),
    "true_positives": tp,
    "false_positives": fp,
    "false_negatives": fn,
    "true_negatives": tn,
    "precision_pct": 100 * precision,
    "recall_pct": 100 * recall,
    "lift_over_random": precision / fraud_prevalence
}])

display(validation_metrics)

In [0]:
display(
    history_df.loc[
        history_df["epoch"] == best_epoch,
        ["epoch", "val_numeric", "val_categorical", "val_kl"]
    ]
)

In [0]:
# Freeze the threshold chosen on validation data.
final_threshold = float(threshold)

# Score the test set using the already-trained model.
test_scores = calculate_anomaly_scores(
    X_test,
    test_num,
    test_cat,
    test_num_mask,
    test_cat_mask
)

assert len(test_scores) == len(test)
assert np.isfinite(test_scores).all()

test_results = test[
    ["Transaction_ID", "User_ID", "Fraudulent"]
].copy()

test_results["anomaly_score"] = test_scores
test_results["predicted_fraud"] = (
    test_scores >= final_threshold
).astype(int)

actual_test = test_results["Fraudulent"].to_numpy()
predicted_test = test_results["predicted_fraud"].to_numpy()

tp = int(((predicted_test == 1) & (actual_test == 1)).sum())
fp = int(((predicted_test == 1) & (actual_test == 0)).sum())
fn = int(((predicted_test == 0) & (actual_test == 1)).sum())
tn = int(((predicted_test == 0) & (actual_test == 0)).sum())

precision = tp / (tp + fp) if tp + fp else np.nan
recall = tp / (tp + fn) if tp + fn else np.nan
prevalence = actual_test.mean()

test_metrics = pd.DataFrame([{
    "threshold": final_threshold,
    "transactions": len(test_results),
    "alerts": tp + fp,
    "alert_rate_pct": 100 * predicted_test.mean(),
    "true_positives": tp,
    "false_positives": fp,
    "false_negatives": fn,
    "true_negatives": tn,
    "precision_pct": 100 * precision,
    "recall_pct": 100 * recall,
    "lift_over_random": (
        precision / prevalence if prevalence > 0 else np.nan
    )
}])

display(test_metrics)